# Loader Files


Aqui está a documentação didática da classe **FilesPayloadBuilder**, estruturada para facilitar o entendimento de desenvolvedores e integradores.

---

## 1. Visão Geral
A classe `FilesPayloadBuilder` atua como um **filtro de segurança e transformador de dados** na camada de entrada da sua API. Quando um usuário envia arquivos através do FastAPI, eles chegam como objetos `UploadFile`. 

Esta classe garante que esses arquivos não "quebrem" o servidor por serem grandes demais ou por possuírem formatos maliciosos ou não suportados. Após a validação, ela transforma os dados brutos em um dicionário Python (payload) fácil de ser manipulado por outras partes do sistema, como serviços de processamento de imagem ou bancos de dados.

---

## 2. Fluxo de Execução
O fluxo de processamento de um arquivo segue este caminho lógico:

1.  **Recebimento**: A API recebe uma lista de arquivos via formulário (Multipart).
2.  **Leitura Assíncrona**: O conteúdo do arquivo é lido da memória ou disco temporário de forma não bloqueante (`await f.read()`).
3.  **Verificação de Tipo**: O sistema confere se o `content_type` (ex: image/png) está na lista de permitidos.
4.  **Verificação de Peso**: O tamanho total dos bytes lidos é comparado com o limite configurado (ex: 10MB).
5.  **Estruturação**: Se tudo estiver correto, os dados são organizados em um dicionário. Se algo falhar, uma exceção HTTP é disparada imediatamente, interrompendo o processo para proteger o servidor.



---

## 3. Resumo de Métodos

| Método | Tipo | Descrição Breve |
| :--- | :--- | :--- |
| `__init__` | Construtor | Configura as regras de negócio (tamanho máximo e tipos aceitos). |
| `build_images_payload` | Corrotina (async) | Valida a lista de arquivos e gera a estrutura de dados final. |

---

## 4. Arquitetura e Insights
* **Segurança Reativa**: Ao validar o tamanho e o tipo MIME logo na entrada, você evita ataques de "Denial of Service" (DoS) por upload de arquivos gigantes.
* **Uso de Set para Performance**: A variável `allowed_types` é convertida para um `set`. Em Python, verificar se um item existe em um `set` é muito mais rápido do que em uma `list` ou `tuple`, especialmente se a lista de formatos crescer.
* **Integração com FastAPI**: O uso de `HTTPException` permite que os erros de validação sejam retornados diretamente ao usuário final com códigos de status apropriados (400 Bad Request), sem a necessidade de blocos try/except extras na rota da API.

---

## 5. Detalhamento da Classe

### Classe FilesPayloadBuilder

**Descrição**
Responsável por validar a integridade de arquivos enviados via upload e prepará-los para o processamento interno. Ela centraliza as regras de "o que pode entrar" no sistema em termos de arquivos de mídia.

**Argumentos**
* `max_mb` (int): O tamanho máximo permitido para cada arquivo individual em Megabytes.
* `allowed_types` (Iterable[str]): Uma coleção de strings representando os tipos MIME aceitos (ex: `["image/jpeg", "application/pdf"]`).

---

### Métodos

#### 1. build_images_payload
**Descrição**
Este é o coração da classe. Ele percorre cada arquivo enviado, lê seu conteúdo binário e verifica se ele atende aos requisitos de segurança definidos no construtor.

**Argumentos**
* `files` (List[UploadFile]): Uma lista de objetos de arquivo originados de um endpoint FastAPI.

**Retornos**
* `list[dict]`: Uma lista de dicionários, onde cada dicionário contém:
    * `filename`: Nome original do arquivo.
    * `content_type`: Formato do arquivo.
    * `size_bytes`: Tamanho real em bytes.
    * `bytes`: O conteúdo binário bruto.

**Raises**
* `HTTPException (400)`: Disparado se o formato do arquivo não for permitido.
* `HTTPException (400)`: Disparado se o arquivo exceder o limite de Megabytes configurado.

**Exemplos**
```python
builder = FilesPayloadBuilder(max_mb=2, allowed_types=["image/png"])

# Em uma rota FastAPI
@app.post("/upload")
async def upload_image(files: List[UploadFile]):
    payload = await builder.build_images_payload(files)
    return {"status": "sucesso", "arquivos_processados": len(payload)}
```